# Nepali Currency Classifier: testing the from-scratch CNN
Run the cells from top to bottom. Use a **CPU runtime** (no GPU needed).
You need the CNN checkpoint file (`currency_cnn.pth`) on your laptop.

In [ ]:
# Cell 1: install the speech library
!pip install -q gTTS

In [ ]:
# Cell 2: imports, the CNN class, and loading my saved model
import os, io
import torch
from torch import nn
from torchvision import transforms
from PIL import Image, ImageOps
from collections import Counter
from IPython.display import display, Audio
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# Same CNN class as my from-scratch training notebook (layers must match exactly)
class CurrencyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
        self.block_1 = block(3, 16)
        self.block_2 = block(16, 32)
        self.block_3 = block(32, 64)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 16 * 16, 7),
        )

    def forward(self, x):
        return self.classifier(self.block_3(self.block_2(self.block_1(x))))

MODEL_PATH = "currency_cnn.pth"
if not os.path.exists(MODEL_PATH):
    up = files.upload()                      # pick the CNN .pth file
    MODEL_PATH = list(up.keys())[0]

# weights_only=False is fine because I trained this file myself
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# works whether the file holds {"model", "classes"} or only the weights
if isinstance(ckpt, dict) and "model" in ckpt:
    state = ckpt["model"]
    class_names = ckpt.get("classes", ['10', '100', '1000', '20', '5', '50', '500'])
else:
    state = ckpt
    class_names = ['10', '100', '1000', '20', '5', '50', '500']

model = CurrencyCNN().to(device)
model.load_state_dict(state)
model.eval()

print("CNN loaded. Classes:", class_names)

In [ ]:
# Cell 3: preprocessing and a single-photo prediction
# The CNN was trained on 128x128 images with NO Normalize step, so we do the same.
preprocess = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

def load_photo(path_or_bytes):
    img = Image.open(path_or_bytes)
    img = ImageOps.exif_transpose(img)       # fix phone photos that are sideways
    return img.convert("RGB")

def predict_one(img):
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(x), dim=1)[0]
    conf, idx = probs.max(0)
    return class_names[idx.item()], conf.item()

In [ ]:
# Cell 4: speech, majority vote, and the 3-photo routine
from gtts import gTTS

def speak(text):
    gTTS(text=text, lang="en").save("answer.mp3")
    display(Audio("answer.mp3", autoplay=True))

PHOTOS_NEEDED = 3

def majority_vote(preds):
    # preds is a list of (label, confidence). Needs MORE than half to agree.
    labels = [p[0] for p in preds]
    winner, count = Counter(labels).most_common(1)[0]
    return winner if count > len(labels) / 2 else None

def identify_note():
    preds = []
    for i in range(PHOTOS_NEEDED):
        while True:
            print(f"Please add photo {i + 1} of {PHOTOS_NEEDED}")
            up = files.upload()
            if up:
                break
            print("No photo received, try again.")
        name = list(up.keys())[0]
        img = load_photo(io.BytesIO(up[name]))
        label, conf = predict_one(img)
        preds.append((label, conf))
        print(f"  Photo {i + 1}: {label} rupees ({conf * 100:.1f}% sure)")

    winner = majority_vote(preds)
    if winner:
        message = f"This is {winner} rupees"
    else:
        message = "The photos do not agree. Please try again"
    print("\nFinal answer:", message)
    speak(message)
    return winner

In [ ]:
# Cell 5: run it (it will ask for 3 photos, one at a time)
final = identify_note()

## Optional: test on all my real photos (for the README comparison)
Mounts Google Drive and checks every photo in `currency_photos/5, 10, 20 ...` one at a time (no voting).
Note: ResNet trained on most of these photos, the CNN never saw them. So this is a fair test for the CNN,
but compare it with ResNet's held-out accuracy, not its training photos.

In [ ]:
# Cell 6 (optional): single-photo accuracy of the CNN on my real photos
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/currency_photos"
total, correct = 0, 0
per_class = {}

for folder in sorted(os.listdir(ROOT)):
    folder_path = os.path.join(ROOT, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            continue
        img = load_photo(os.path.join(folder_path, fname))
        label, conf = predict_one(img)
        total += 1
        ok = (label == folder)
        correct += ok
        c = per_class.setdefault(folder, [0, 0])
        c[0] += ok
        c[1] += 1

print(f"CNN on my real photos: {correct}/{total} = {100 * correct / max(total, 1):.1f}%")
for k in sorted(per_class, key=lambda s: int(s)):
    ok, n = per_class[k]
    print(f"  {k:>5} rupees: {ok}/{n} = {100 * ok / n:.0f}%")